In [1]:
from pyserini.search.lucene import LuceneSearcher
from pyserini.index.lucene import LuceneIndexReader
from pyserini.vectorizer import BM25Vectorizer
import json
from tqdm import tqdm
import re
from utils import *

In [2]:
searcher = LuceneSearcher("hotpotqa_index")
index_reader = LuceneIndexReader("hotpotqa_index")
vectorizer = BM25Vectorizer("hotpotqa_index")

In [3]:
vectorizer_term_to_id = vectorizer.term_to_index
vectorizer_id_to_term = {v: k for k, v in vectorizer_term_to_id.items()}

In [4]:
def extract_query_vector(query):
    query_vector = {}
    vector = vectorizer.get_query_vector(query)
    for data, id in zip(vector.data, vector.indices):
        term = vectorizer_id_to_term[id]
        query_vector[term] = int(data)
    return query_vector

In [5]:
with open('data/train_data.jsonl') as f:
    train_data = [json.loads(line) for line in f]
with open('data/validation_data.jsonl') as f:
    validation_data = [json.loads(line) for line in f]

In [6]:
queries = [item['query'] for item in train_data] + [item['query'] for item in validation_data]
queries = list(set(queries))  # unique queries
len(queries)

97399

In [7]:
train_query_dict = {}
with open('data/train_query_vectors_dict.json', 'w') as f:
    for query in tqdm(queries):
        query_vector = extract_query_vector(query)
        train_query_dict[text_to_id(query)] = query_vector
    json.dump(train_query_dict, f, indent=4)

100%|██████████| 97399/97399 [22:20<00:00, 72.66it/s]


In [23]:
index_reader.dump_documents_BM25("data/hotpotqa_bm25_vectors.jsonl")

100%|██████████| 509115/509115 [07:29<00:00, 1131.73it/s]


In [24]:
with open("data/hotpotqa_bm25_vectors.jsonl") as f, open("data/train_doc_vectors_dict.json", "w") as f_out:
    output_dict = {}
    lines = f.readlines()
    for i, line in tqdm(enumerate(lines), desc="Processing BM25 vectors", total=len(lines)):
        bm25_data = json.loads(line)
        output_dict[bm25_data['id']] = bm25_data['vector']
    json.dump(output_dict, f_out, indent=4)

Processing BM25 vectors: 100%|██████████| 509115/509115 [00:11<00:00, 42700.58it/s]


In [31]:
id_to_term = {}
term_to_id = {}
count = 0
idx = 0
count_lim = 10
percent_doc_lim = 0.1
min_term_length = 4
for t in index_reader.terms():
    if t.df*(1+percent_doc_lim) >= t.cf:
        count += 1
        continue
    if not re.fullmatch(r"[A-Za-z]+", t.term):
        count += 1
        continue
    if t.cf < count_lim:
        count += 1
        continue
    if len(t.term) < min_term_length:
        count += 1
        continue
    id_to_term[idx] = t.term
    term_to_id[t.term] = idx
    idx += 1
    print(f"term: {t.term}, df: {t.df}, cf: {t.cf}")

term: aaaa, df: 9, cf: 10
term: aaberg, df: 7, cf: 12
term: aachen, df: 68, cf: 106
term: aacta, df: 67, cf: 162
term: aadi, df: 11, cf: 14
term: aafc, df: 50, cf: 80
term: aaja, df: 10, cf: 13
term: aalborg, df: 47, cf: 78
term: aaliyah, df: 58, cf: 82
term: aalst, df: 6, cf: 11
term: aalto, df: 20, cf: 55
term: aamer, df: 8, cf: 10
term: aamina, df: 9, cf: 10
term: aamir, df: 73, cf: 92
term: aang, df: 14, cf: 17
term: aapa, df: 7, cf: 12
term: aarau, df: 11, cf: 14
term: aardman, df: 31, cf: 41
term: aardvark, df: 13, cf: 17
term: aargau, df: 18, cf: 24
term: aarhu, df: 85, cf: 172
term: aaron, df: 1186, cf: 1322
term: aarp, df: 11, cf: 14
term: aarseth, df: 9, cf: 11
term: aarti, df: 14, cf: 23
term: aarushi, df: 5, cf: 10
term: ababa, df: 68, cf: 90
term: abaco, df: 11, cf: 15
term: abacu, df: 17, cf: 24
term: abad, df: 41, cf: 50
term: abaddon, df: 8, cf: 11
term: abadi, df: 8, cf: 12
term: abarth, df: 11, cf: 16
term: abat, df: 23, cf: 33
term: abba, df: 303, cf: 451
term: abban

In [32]:
count, len(id_to_term)

(499029, 45042)

In [33]:
with open("data/hotpotqa_vocab_id_to_term.json", "w") as f:
    json.dump(id_to_term, f, indent=4)

with open("data/hotpotqa_vocab_term_to_id.json", "w") as f:
    json.dump(term_to_id, f, indent=4)

In [34]:
searcher_test = LuceneSearcher("wikiclir_index")
index_reader_test = LuceneIndexReader("wikiclir_index")

In [35]:
with open('data/test_queries.jsonl') as f:
    test_query_data = [json.loads(line) for line in f]

In [36]:
queries = [item['question'] for item in test_query_data]
queries = list(set(queries))  # unique queries
len(queries)

2173

In [ ]:
test_query_dict = {}
with open('data/test_query_vectors_dict.json', 'w') as f:
    for query in tqdm(queries):
        query_vector = extract_query_vector(query)
        test_query_dict[text_to_id(query)] = query_vector
    json.dump(test_query_dict, f, indent=4)

100%|██████████| 2173/2173 [00:53<00:00, 40.34it/s]


In [38]:
index_reader_test.dump_documents_BM25("data/wikiclir_bm25_vectors.jsonl")

100%|██████████| 126916/126916 [04:03<00:00, 520.64it/s]


In [39]:
with open("data/wikiclir_bm25_vectors.jsonl") as f, open("data/test_doc_vectors_dict.json", "w") as f_out:
    output_dict = {}
    lines = f.readlines()
    for i, line in tqdm(enumerate(lines), desc="Processing BM25 vectors", total=len(lines)):
        bm25_data = json.loads(line)
        output_dict[bm25_data['id']] = bm25_data['vector']
    json.dump(output_dict, f_out, indent=4)

Processing BM25 vectors: 100%|██████████| 126916/126916 [00:05<00:00, 23721.53it/s]
